In [1]:
# =========================
# 1. Imports
# =========================
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm
from scipy.sparse import hstack, csr_matrix

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score


In [2]:
# =========================
# 2. Load and prepare data
# =========================
df = pd.read_csv("data/dataset.csv")

# Keep only binary outcome rows
df = df[df["state"].isin(["successful", "failed"])].copy()

# Build text input
df["text"] = df["name"].fillna("") + " " + df["blurb"].fillna("")

# Binary label: 1 = successful, 0 = failed
df["label"] = (df["state"] == "successful").astype(int)

# Safe structured features only (NO leakage)
structured_cols = [
    "goal_usd",
    "duration",
    "country",
    "category.parent_name",
    "category.name",
    "CCI_index",
    "launched_at-year",
    "launched_at-month"
]

df = df[["text", "label"] + structured_cols].dropna().reset_index(drop=True)

print(df.shape)
df.head()


(106512, 10)


,text,label,goal_usd,duration,country,category.parent_name,category.name,CCI_index,launched_at-year,launched_at-month
0,Help get Das Good seasonings and sauces to sup...,0,75000.0,60,US,Food,Restaurants,101.2287,2018,12
1,"""The Applicant: Interviews Are Hell"" A web ser...",0,6000.0,30,US,Film & Video,Webseries,101.2478,2016,12
2,5 Reasons to Hate Christmas A romantic comedy ...,0,10000.0,30,US,Film & Video,Webseries,101.5817,2018,4
3,The Drums of Atlant The Drums of Atlant is an ...,0,20000.0,30,US,Film & Video,Science Fiction,101.4410,2019,4
4,"Ordinary Beauty ""...a real photographer will m...",0,985.0,14,US,Photography,Photobooks,100.8538,2015,7


In [3]:
# =========================
# 3. Train/test split first
#    Important: avoid extracting embeddings
#    on the whole dataset before splitting
# =========================
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (85209, 10)
Test shape: (21303, 10)


In [4]:
# =========================
# 4. Load frozen DistilBERT
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)
bert_model.eval()  # frozen mode


Using device: cpu


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [5]:
# =========================
# 5. Mean pooling function
# =========================
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


In [6]:
# =========================
# 6. Embedding extraction
# =========================
def get_frozen_embeddings(texts, batch_size=32, max_length=128):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = bert_model(**inputs)
            pooled = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])

        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)


# Extract embeddings separately for train and test
X_train_text = get_frozen_embeddings(train_df["text"].tolist(), batch_size=32, max_length=128)
X_test_text = get_frozen_embeddings(test_df["text"].tolist(), batch_size=32, max_length=128)

print("Train text embeddings shape:", X_train_text.shape)
print("Test text embeddings shape:", X_test_text.shape)


100%|██████████| 666/666 [02:07<00:00,  5.21it/s]

Train text embeddings shape: (85209, 768)
Test text embeddings shape: (21303, 768)


In [7]:
# =========================
# 7. Preprocess structured features
# =========================
numeric_features = ["goal_usd", "duration", "CCI_index", "launched_at-year", "launched_at-month"]
categorical_features = ["country", "category.parent_name", "category.name"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train_struct = preprocessor.fit_transform(train_df[structured_cols])
X_test_struct = preprocessor.transform(test_df[structured_cols])

print("Train structured shape:", X_train_struct.shape)
print("Test structured shape:", X_test_struct.shape)


Train structured shape: (85209, 171)
Test structured shape: (21303, 171)


In [12]:
# =========================
# 8. Build two feature sets
# =========================

# A) Frozen DistilBERT only
X_train_frozen_only = csr_matrix(X_train_text)
X_test_frozen_only = csr_matrix(X_test_text)

# B) Frozen DistilBERT + structured features
X_train_frozen_plus = hstack([csr_matrix(X_train_text), X_train_struct])
X_test_frozen_plus = hstack([csr_matrix(X_test_text), X_test_struct])

y_train = train_df["label"].values
y_test = test_df["label"].values

print("Frozen only train shape:", X_train_frozen_only.shape)
print("Frozen only test shape:", X_test_frozen_only.shape)
print("Frozen + structured train shape:", X_train_frozen_plus.shape)
print("Frozen + structured test shape:", X_test_frozen_plus.shape)

Frozen only train shape: (85209, 768)
Frozen only test shape: (21303, 768)
Frozen + structured train shape: (85209, 939)
Frozen + structured test shape: (21303, 939)


In [13]:
# =========================
# 9. Train and evaluate helper
# =========================
def evaluate_logreg(X_train, X_test, y_train, y_test, model_name):
    clf = LogisticRegression(
        max_iter=2000,
        random_state=42
    )
    
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    print(f"\n{model_name} results")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")

    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    return {
        "Model": model_name,
        "accuracy": acc,
        "f1": f1,
        "precision": prec,
        "recall": rec
    }

In [14]:
# =========================
# 10. Run both frozen experiments
# =========================
frozen_only_results = evaluate_logreg(
    X_train_frozen_only,
    X_test_frozen_only,
    y_train,
    y_test,
    "Frozen DistilBERT only"
)

frozen_plus_results = evaluate_logreg(
    X_train_frozen_plus,
    X_test_frozen_plus,
    y_train,
    y_test,
    "Frozen DistilBERT + structured features"
)


Frozen DistilBERT only results
Accuracy : 0.7421
F1       : 0.7896
Precision: 0.7597
Recall   : 0.8221

Classification report:
              precision    recall  f1-score   support

           0       0.71      0.63      0.67      8757
           1       0.76      0.82      0.79     12546

    accuracy                           0.74     21303
   macro avg       0.74      0.72      0.73     21303
weighted avg       0.74      0.74      0.74     21303


Confusion matrix:
[[ 5494  3263]
 [ 2232 10314]]

Frozen DistilBERT + structured features results
Accuracy : 0.8056
F1       : 0.8341
Precision: 0.8384
Recall   : 0.8298

Classification report:
              precision    recall  f1-score   support

           0       0.76      0.77      0.77      8757
           1       0.84      0.83      0.83     12546

    accuracy                           0.81     21303
   macro avg       0.80      0.80      0.80     21303
weighted avg       0.81      0.81      0.81     21303


Confusion matrix:
[[ 6

In [15]:
# =========================
# 11. Compare with fine-tuned result
# =========================
finetuned_results = {
    "Model": "Fine-tuned DistilBERT (text only)",
    "accuracy": 0.7681,
    "f1": 0.8233,
    "precision": 0.7601,
    "recall": 0.8980
}

comparison_df = pd.DataFrame([
    frozen_only_results,
    frozen_plus_results,
    finetuned_results
])

comparison_df

,Model,accuracy,f1,precision,recall
0,Frozen DistilBERT only,0.742055,0.789649,0.759667,0.822095
1,Frozen DistilBERT + structured features,0.805614,0.834114,0.838447,0.829826
2,Fine-tuned DistilBERT (text only),0.768100,0.823300,0.760100,0.898000
